# Kaggle: Predicción de precios de portátiles (Random Forest) — versión paso a paso (sin funciones)
Mismo flujo que el notebook anterior, pero haciendo el **preprocesado directamente sobre el DataFrame** para que se vea el **paso a paso**.

Métrica: **RMSE** (más bajo = mejor).

In [ ]:
import numpy as np
import pandas as pd
import re
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor

# Chequeador Kaggle 
import urllib.request
from PIL import Image

RANDOM_STATE = 42


## 1) Carga de datos

In [4]:
train = pd.read_csv('data/train.csv')
test  = pd.read_csv('data/test.csv')
sample = pd.read_csv('data/sample_submission.csv')

train.shape, test.shape, sample.shape

((912, 13), (391, 12), (391, 2))

## 2) Preprocesado paso a paso (limpieza + feature engineering)

### 2.1) Copias de trabajo

In [5]:
train_p = train.copy()
test_p  = test.copy()


### 2.2) RAM: `'8GB' → 8`

In [6]:
for df in (train_p, test_p):
    df['Ram_GB'] = df['Ram'].str.extract(r'(\d+)').astype(float)

train_p[['Ram', 'Ram_GB']].head()

,Ram,Ram_GB
0,8GB,8.0
1,16GB,16.0
2,8GB,8.0
3,8GB,8.0
4,4GB,4.0


### 2.3) Peso: `'1.86kg' → 1.86`

In [7]:
for df in (train_p, test_p):
    df['Weight_kg'] = df['Weight'].str.lower().str.replace('kg','', regex=False).astype(float)

train_p[['Weight', 'Weight_kg']].head()

,Weight,Weight_kg
0,1.86kg,1.86
1,2.59kg,2.59
2,2.04kg,2.04
3,1.34kg,1.34
4,2.25kg,2.25


### 2.4) Resolución: X/Y + IPS/Touch + PPI

In [8]:
def add_screen_features(df):
    df['IPS'] = df['ScreenResolution'].str.contains(r'\bIPS\b', case=False, na=False).astype(int)
    df['Touchscreen'] = df['ScreenResolution'].str.contains('touch', case=False, na=False).astype(int)

    res = df['ScreenResolution'].str.extract(r'(?P<X_res>\d{3,4})\s*x\s*(?P<Y_res>\d{3,4})')
    df['X_res'] = pd.to_numeric(res['X_res'], errors='coerce')
    df['Y_res'] = pd.to_numeric(res['Y_res'], errors='coerce')

    df['ppi'] = np.sqrt(df['X_res']**2 + df['Y_res']**2) / df['Inches']
    return df

train_p = add_screen_features(train_p)
test_p  = add_screen_features(test_p)

train_p[['ScreenResolution','X_res','Y_res','IPS','Touchscreen','ppi']].head()

,ScreenResolution,X_res,Y_res,IPS,Touchscreen,ppi
0,Full HD 1920x1080,1920,1080,0,0,141.211998
1,Full HD 1920x1080,1920,1080,0,0,141.211998
2,Full HD 1920x1080,1920,1080,0,0,141.211998
3,1440x900,1440,900,0,0,127.677940
4,Full HD 1920x1080,1920,1080,0,0,141.211998


### 2.5) CPU: marca, familia, GHz

In [9]:
def add_cpu_features(df):
    cpu = df['Cpu'].astype(str)

    df['Cpu_brand'] = np.select(
        [cpu.str.contains('intel', case=False, na=False),
         cpu.str.contains('amd', case=False, na=False)],
        ['Intel', 'AMD'],
        default='Other'
    )

    df['Cpu_family'] = (
        cpu.str.extract(r'(Core i[3579]|Ryzen\s*[3579]|Celeron|Pentium|Atom|Xeon|M[357])', flags=re.I)[0]
        .fillna(cpu.str.split().str[:2].str.join(' '))
    )

    df['Cpu_ghz'] = pd.to_numeric(cpu.str.extract(r'(\d+(?:\.\d+)?)\s*GHz', flags=re.I)[0], errors='coerce')
    return df

train_p = add_cpu_features(train_p)
test_p  = add_cpu_features(test_p)

train_p[['Cpu','Cpu_brand','Cpu_family','Cpu_ghz']].head()

,Cpu,Cpu_brand,Cpu_family,Cpu_ghz
0,Intel Core i3 6006U 2GHz,Intel,Core i3,2.0
1,Intel Core i7 6700HQ 2.6GHz,Intel,Core i7,2.6
2,Intel Core i7 7500U 2.7GHz,Intel,Core i7,2.7
3,Intel Core i5 1.8GHz,Intel,Core i5,1.8
4,Intel Core i3 6006U 2.0GHz,Intel,Core i3,2.0


### 2.6) GPU: marca

In [10]:
g_train = train_p['Gpu'].astype(str).str.lower()
g_test  = test_p['Gpu'].astype(str).str.lower()

train_p['Gpu_brand'] = np.select(
    [g_train.str.contains('nvidia', na=False),
     g_train.str.contains('amd|radeon', na=False),
     g_train.str.contains('intel', na=False)],
    ['Nvidia', 'AMD', 'Intel'],
    default='Other'
)

test_p['Gpu_brand'] = np.select(
    [g_test.str.contains('nvidia', na=False),
     g_test.str.contains('amd|radeon', na=False),
     g_test.str.contains('intel', na=False)],
    ['Nvidia', 'AMD', 'Intel'],
    default='Other'
)

train_p[['Gpu','Gpu_brand']].head()

,Gpu,Gpu_brand
0,Intel HD Graphics 520,Intel
1,Nvidia GeForce GTX 960<U+039C>,Nvidia
2,Nvidia GeForce 930MX,Nvidia
3,Intel HD Graphics 6000,Intel
4,AMD Radeon R5 M430,AMD


### 2.7) Memory: SSD/HDD/Flash/Hybrid + total (GB)

In [11]:
def to_gb(series):
    num = pd.to_numeric(series.str.extract(r'(\d+(?:\.\d+)?)')[0], errors='coerce')
    unit = series.str.extract(r'(TB|GB)', flags=re.I)[0].str.upper()
    return np.where(unit == 'TB', num * 1024, num)

def add_memory_features(df):
    mem = df['Memory'].astype(str).str.replace(' ', '', regex=False)

    p1 = mem.str.split('+').str[0]
    p2 = mem.str.split('+').str[1].fillna('')

    def amount_if(part, keyword):
        mask = part.str.contains(keyword, case=False, na=False)
        gb = to_gb(part)
        return np.where(mask, gb, 0.0)

    df['SSD_GB'] = amount_if(p1, 'SSD') + amount_if(p2, 'SSD')
    df['HDD_GB'] = amount_if(p1, 'HDD') + amount_if(p2, 'HDD')
    df['Flash_GB'] = amount_if(p1, 'Flash') + amount_if(p2, 'Flash')
    df['Hybrid_GB'] = amount_if(p1, 'Hybrid') + amount_if(p2, 'Hybrid')
    df['Total_Memory_GB'] = df['SSD_GB'] + df['HDD_GB'] + df['Flash_GB'] + df['Hybrid_GB']
    return df

train_p = add_memory_features(train_p)
test_p  = add_memory_features(test_p)

train_p[['Memory','SSD_GB','HDD_GB','Flash_GB','Hybrid_GB','Total_Memory_GB']].head()

,Memory,SSD_GB,HDD_GB,Flash_GB,Hybrid_GB,Total_Memory_GB
0,256GB SSD,256.0,0.0,0.0,0.0,256.0
1,1TB HDD,0.0,1024.0,0.0,0.0,1024.0
2,1TB HDD,0.0,1024.0,0.0,0.0,1024.0
3,128GB Flash Storage,0.0,0.0,128.0,0.0,128.0
4,1TB HDD,0.0,1024.0,0.0,0.0,1024.0


### 2.8) Drop de columnas originales “sucias”

In [12]:
drop_cols = ['Ram','Weight','ScreenResolution','Cpu','Memory','Gpu','Product']
train_p = train_p.drop(columns=drop_cols)
test_p  = test_p.drop(columns=drop_cols)

train_p.head()

,laptop_ID,Company,TypeName,Inches,OpSys,Price_in_euros,Ram_GB,Weight_kg,IPS,Touchscreen,...,ppi,Cpu_brand,Cpu_family,Cpu_ghz,Gpu_brand,SSD_GB,HDD_GB,Flash_GB,Hybrid_GB,Total_Memory_GB
0,755,HP,Notebook,15.6,Windows 10,539.00,8.0,1.86,0,0,...,141.211998,Intel,Core i3,2.0,Intel,256.0,0.0,0.0,0.0,256.0
1,618,Dell,Gaming,15.6,Windows 10,879.01,16.0,2.59,0,0,...,141.211998,Intel,Core i7,2.6,Nvidia,0.0,1024.0,0.0,0.0,1024.0
2,909,HP,Notebook,15.6,Windows 10,900.00,8.0,2.04,0,0,...,141.211998,Intel,Core i7,2.7,Nvidia,0.0,1024.0,0.0,0.0,1024.0
3,2,Apple,Ultrabook,13.3,macOS,898.94,8.0,1.34,0,0,...,127.677940,Intel,Core i5,1.8,Intel,0.0,0.0,128.0,0.0,128.0
4,286,Dell,Notebook,15.6,Linux,428.00,4.0,2.25,0,0,...,141.211998,Intel,Core i3,2.0,AMD,0.0,1024.0,0.0,0.0,1024.0


## 3) Nulos + One-Hot (stats del train) + alineación

In [13]:
y = train_p['Price_in_euros']
X = train_p.drop(columns=['Price_in_euros'])

num_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object','bool']).columns.tolist()

# Imputación
med = X[num_cols].median()
X[num_cols] = X[num_cols].fillna(med)
test_p[num_cols] = test_p[num_cols].fillna(med)

for c in cat_cols:
    mode_val = X[c].mode(dropna=True)[0]
    X[c] = X[c].fillna(mode_val)
    test_p[c] = test_p[c].fillna(mode_val)

# One-hot + alinear
X_enc = pd.get_dummies(X, columns=cat_cols, drop_first=True)
test_enc = pd.get_dummies(test_p, columns=cat_cols, drop_first=True)
test_enc = test_enc.reindex(columns=X_enc.columns, fill_value=0)

X_enc.shape, test_enc.shape

((912, 68), (391, 68))

## 4) Validación local (RMSE)

In [14]:
X_train, X_val, y_train, y_val = train_test_split(
    X_enc, y, test_size=0.2, random_state=RANDOM_STATE
)

rf = RandomForestRegressor(
    n_estimators=600,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf.fit(X_train, y_train)

pred_train = rf.predict(X_train)
pred_val = rf.predict(X_val)

rmse_train = np.sqrt(mean_squared_error(y_train, pred_train))
rmse_val   = np.sqrt(mean_squared_error(y_val, pred_val))

print(f"RMSE train: {rmse_train:.3f}")
print(f"RMSE val:   {rmse_val:.3f}")
print(f"Diferencia: {rmse_val - rmse_train:.3f}")

RMSE train: 104.024
RMSE val:   334.787
Diferencia: 230.763


## 5) Entrenamiento final + submission

In [15]:
rf_final = RandomForestRegressor(
    n_estimators=800,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_final.fit(X_enc, y)

test_preds = rf_final.predict(test_enc)

submission = pd.DataFrame({
    'laptop_ID': test_p['laptop_ID'].values,
    'Price_in_euros': test_preds
})

submission.head(), submission.shape

(   laptop_ID  Price_in_euros
 0        209     1497.186387
 1       1281      295.607812
 2       1168      391.528162
 3       1231      999.871175
 4       1020     1110.691487,
 (391, 2))

## 6) Chequeador Kaggle

In [16]:
def chequeador(df_to_submit):
    if df_to_submit.shape == sample.shape:
        if df_to_submit.columns.all() == sample.columns.all():
            if df_to_submit.laptop_ID.all() == sample.laptop_ID.all():
                print("You're ready to submit!")
                df_to_submit.to_csv("submission.csv", index=False)
                urllib.request.urlretrieve(
                    "https://www.mihaileric.com/static/evaluation-meme-e0a350f278a36346e6d46b139b1d0da0-ed51e.jpg",
                    "gfg.png"
                )
                img = Image.open("gfg.png")
                img.show()
            else:
                print("Check the ids and try again")
        else:
            print("Check the names of the columns and try again")
    else:
        print("Check the number of rows and/or columns and try again")
        print("\nMensaje secreto del TA: has tocado filas de test.csv :(")


In [17]:
chequeador(submission)

You're ready to submit!
